<a href="https://colab.research.google.com/github/MoulendraBalaji/Flyrank_ML_Works/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

Load the starter dataset, then define the slice my lane actually operates on.

In [1]:
import os
import pandas as pd

# The starter dataset lives at the repo root; Colab users may need to upload or mount it.
candidates = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "/content/content_refresh_anonymized.csv",
]
data_path = next((p for p in candidates if os.path.exists(p)), candidates[0])

df = pd.read_csv(data_path)

# Lane 2 review universe: pages with enough demand to justify an editor's hour.
review_universe = df[df["impressions_90d"] >= 100].copy()

# Proxy label (defined and justified in Section 2).
review_universe["declining_with_demand"] = (
    review_universe["trend_direction"] == "down"
).astype(int)

print(f"Loaded {len(df):,} content items; review universe = {len(review_universe):,} pages")

Loaded 30,000 content items; review universe = 22,006 pages


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring.** My lane is **Lane 2 — Refresh / Content Opportunity Scoring**.

The decision is *"which pages should a content editor review first?"* — an **ordering**
question, not a yes/no question. Each page gets a **priority score**, and the score feeds a
**ranked review queue**. We do not need one verdict per page; we need the best ordering under a
fixed human capacity (an editor can realistically review 20–50 pages a day).

A plain classifier is an *input* here, not the deliverable: even if every declining page were
flagged perfectly, that still gives no order. The model's job is to rank the same pages so the
*front* of the list is where the real opportunities are. That is why this maps to
scoring/ranking, evaluated at the top K.

In [2]:
lane = "Lane 2: Refresh / Content Opportunity Scoring"
task_type = "Ranking / scoring"

print(f"Lane     : {lane}")
print(f"Task type: {task_type}")
print(f"Output   : a priority score per page -> an ordered review queue")

# Why classification alone is not enough: the blunt rule has no ordering.
blunt_rule_flags = int((df["trend_direction"] == "down").sum())
print(f"\nA blunt rule (\"refresh every declining page\") flags {blunt_rule_flags:,} of {len(df):,} pages")
print("...far beyond editor capacity. The decision is WHICH FIRST -> we need an ordering, not a filter.")

Lane     : Lane 2: Refresh / Content Opportunity Scoring
Task type: Ranking / scoring
Output   : a priority score per page -> an ordered review queue

A blunt rule ("refresh every declining page") flags 16,262 of 30,000 pages
...far beyond editor capacity. The decision is WHICH FIRST -> we need an ordering, not a filter.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (proxy): `declining_with_demand`** — a page that is *losing visibility* while *still
having demand*: the profile of a page worth refreshing first. It is 1 when
`trend_direction == "down"` **and** `impressions_90d >= 100`, else 0.

**Where the label comes from — observed measurements, not a copied rule.** The `down` flag is a
transparent bucket over two *measured* windows: last-30-day impressions versus the previous 30
days (down = more than 20% lower, per the documented bucket in the data dictionary). The demand
test uses *measured* impressions over the trailing 90 days. Nothing here is a product score or a
hand-written decision flag — it is a derivation of what actually happened.

**Honesty note (why it is a proxy, not the real outcome):** on this single-snapshot starter
slice the label describes the page's *current* state, not a *future* outcome. The stronger
version — which the warehouse makes possible — is a future-window label: features from the prior
90 days → decline or recovery over the next 30 days, measured after a clear decision point. On
the starter data I can only claim a *directional* proxy, and I will say so in every downstream
notebook.

Because the label is built from `trend_direction` / `trend_pct`, those columns can **never** be
model features — using them would leak the answer.

In [3]:
n_pos = int(review_universe["declining_with_demand"].sum())
n_neg = len(review_universe) - n_pos
print(f"declining_with_demand = 1 : {n_pos:,} pages  ({review_universe['declining_with_demand'].mean():.1%} of universe)")
print(f"declining_with_demand = 0 : {n_neg:,} pages")

print("\nLabel source: transparent derivation from OBSERVED measurements")
print("  - 'down'  -> measured last-30d impressions vs previous 30d, dropped > 20% (documented bucket)")
print("  - demand  -> measured impressions_90d >= 100")

leaky = ["trend_direction", "trend_pct", "is_declining_label"]
print("\nNever features (they encode the label):", ", ".join(leaky))

declining_with_demand = 1 : 13,152 pages  (59.8% of universe)
declining_with_demand = 0 : 8,854 pages

Label source: transparent derivation from OBSERVED measurements
  - 'down'  -> measured last-30d impressions vs previous 30d, dropped > 20% (documented bucket)
  - demand  -> measured impressions_90d >= 100

Never features (they encode the label): trend_direction, trend_pct, is_declining_label


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50** — of the top 50 pages the queue sends to an editor first, how many are
genuinely `declining_with_demand`? Top-K precision matches how the output is *actually used*: the
editor works down the list until capacity runs out, so what matters is that the **front** of the
queue is right — not raw accuracy over the whole ranking.

Numbers to beat (computed below):

- **Random / base rate:** 59.8% of the review universe is `declining_with_demand` — a blind
  top-50 would be right about 30 of 50 times.
- **A naive fixed rule ("stalest first"):** 68% Precision@50 — 34 of 50.
- **Good means:** a model that clears these, ideally ≥ 80% (40 of 50), on client-held-out data —
  never on pages the model already saw. I will also track Precision@20 for smaller teams.

In [4]:
base_rate = review_universe["declining_with_demand"].mean()

naive_top50 = review_universe.nlargest(50, "days_since_last_update")
naive_p50 = naive_top50["declining_with_demand"].mean()

print(f"Label base rate in the review universe : {base_rate:.1%}   <- blind top-50 would land here")
print(f"Naive 'stalest first' rule  Precision@50: {naive_p50:.0%}   ({int(round(naive_p50 * 50))}/50 correct)")

print(f"\nSuccess metric : Precision@50, must beat the {naive_p50:.0%} rule baseline")
print("Good = the first 50 pages an editor sees are mostly real decline-with-demand candidates,")
print("       measured on clients the model has never trained on.")

Label base rate in the review universe : 59.8%   <- blind top-50 would land here
Naive 'stalest first' rule  Precision@50: 68%   (34/50 correct)

Success metric : Precision@50, must beat the 68% rule baseline
Good = the first 50 pages an editor sees are mostly real decline-with-demand candidates,
       measured on clients the model has never trained on.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (one pseudonymized `content_id`) with its trailing-90-day
measurements. This is the grain for the whole lane: every score, feature, and label is a property
of *a page*, so the model and the queue both operate on one row per page.

The slice below is my lane's review universe — the 22,006 pages with measurable demand
(`impressions_90d >= 100`) that could plausibly justify an editor's hour. The last column,
`declining_with_demand`, shows what the target looks like row by row.

In [5]:
cols = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction",
    "declining_with_demand",
]

print(f"Unit of analysis: one row = one content page. Universe shape: {review_universe.shape[0]:,} rows x {review_universe.shape[1]} cols")
review_universe[cols].head(10)

Unit of analysis: one row = one content page. Universe shape: 22,006 rows x 45 cols


,content_id,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,word_count,trend_direction,declining_with_demand
0,content_304f48230142,3803,29,0.76,10.6,187,20,3221.0,down,1
1,content_a1fb4e703a9e,15320,7,0.05,20.3,445,25,2481.0,down,1
2,content_9aa793d4d895,12581,11,0.09,36.5,141,20,3515.0,down,1
3,content_331d6c4de07b,11751,58,0.49,6.2,463,22,NaN,stable,0
4,content_d99b7a2d90ca,19140,24,0.13,44.0,263,14,2803.0,down,1
5,content_d4084a4bc775,3970,1,0.03,8.5,147,20,3080.0,down,1
7,content_a63219c6e95a,1724,1,0.06,21.2,445,22,NaN,stable,0
8,content_5e6c160719bc,32574,29,0.09,46.0,90,20,3807.0,down,1
9,content_c27558df2b0c,1240,2,0.16,4.9,257,104,NaN,down,1
10,content_d8ee6cc6d642,20919,324,1.55,2.2,329,104,NaN,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Three things a hand-written if-statement gets wrong:

1. **Rules don't order.** "Refresh every declining page" flags 16,262 of 30,000 pages. A rule
   can filter; it cannot tell you which of those 16,262 to look at *first*. The output has to be
   a continuous priority score, and learning that ordering is exactly what ranking models do.

2. **The pattern is tangled, not a single threshold.** The cross-tabs below show
   decline-with-demand spreads unevenly — and sometimes *backwards* — across single dimensions:
   - **Position:** top-3 pages decline at ~76%, while pages in the deep tiers at ~32% — position
     matters a lot.
   - **Age:** the mature-but-young tier (91–180 days) declines at ~69%, while the oldest tier
     (365+ days) declines at only ~43%. A naive "refresh the oldest pages first" rule would work
     **backwards** — it would prioritize the tier with the least decline.
   No single cutoff (age, position, trend, or word count) separates good candidates from bad
   ones. The signal lives in the *combination*, which is the shape a learned model handles and a
   fixed rule cannot.

3. **Rules freeze at the moment they are tuned.** Thresholds chosen by hand do not re-fit when
   the search landscape shifts; a learned score is re-trained on fresh data and re-ranks itself.

The output stays **decision-support**: an ordering a human still reviews, never an automated
verdict. The starter pipeline already previews the upside — its learned model reached 0.74
Precision@50 versus 0.24 for the hand-written baseline on the 30k slice. We will re-earn that
result on the warehouse with clean validation rather than assuming it carries over.

In [6]:
print("Decline-with-demand rate splits by single dimensions (the mess a fixed rule must ignore):")

print("\nby position_tier:")
print(review_universe.groupby("position_tier")["declining_with_demand"].agg(["count", "mean"]).round(3))

print("\nby age_tier:")
print(review_universe.groupby("age_tier")["declining_with_demand"].agg(["count", "mean"]).round(3))

Decline-with-demand rate splits by single dimensions (the mess a fixed rule must ignore):

by position_tier:
               count   mean
position_tier              
deep             879  0.317
page_1          8633  0.607
page_3_5        6058  0.584
striking        5903  0.626
top_3            533  0.756

by age_tier:
          count   mean
age_tier              
181-365    7650  0.606
31-90       304  0.681
365+       5335  0.427
91-180     8717  0.692


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.